# EllipseNet Final — NumPy-Only Oriented Ellipse Detector

**AML Final Project** | Built from scratch using NumPy only — no PyTorch/TF autograd.

### Architecture in one line
`Conv → LeakyReLU → MaxPool (×4) + Conv1×1 bottlenecks (×2) → Dense head → (S,S,B,6+C) ellipse predictions`

### Notebook map
| Section | Content | Project Step |
|---------|---------|--------------|
| 0 | Imports & all hyperparameters | — |
| 1 | VOC parsing → ellipse annotation conversion | Step 2 |
| 2 | NumPy layers with manual backprop | Step 3 |
| 3 | Adam optimiser from scratch | Step 4 |
| 4 | EllipseNet architecture | Step 3 |
| 5 | Ellipse IoU (Monte Carlo) + loss function | Step 3 |
| 6 | Training loop | Step 4 |
| 7 | Loss curve visualisation | Step 4 |
| 8 | Precision / Recall / mAP evaluation | Step 5 |
| 9 | Prediction visualisation | Step 5 |


## Section 0 — Imports & Hyperparameters
All tuneable constants live here. Use `OVERFIT_SANITY_RUN`, `STAGED_SAMPLES`, and the rotation controls to move from quick checks to larger runs.


In [ ]:
import os, json, time, xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse as MplEllipse

np.random.seed(42)
os.makedirs("figures",  exist_ok=True)
os.makedirs("outputs",  exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

# ── Dataset paths ──────────────────────────────────────────────────────────
VOC_SEARCH_ROOT = "../archive/VOC2012_train_val/VOC2012_train_val"

# ── Image / grid config ────────────────────────────────────────────────────
# 224×224 with five /2 downsampling stages gives a 7×7 grid,
# matching the classic YOLO-style detection grid while staying below 448×448 cost.
IMG_SIZE = 224
S        = 7     # output grid (cells per side)
B        = 2     # anchor ellipses per grid cell
C        = 20    # number of VOC classes

# ── Run mode / training ───────────────────────────────────────────────────
# Use FULL_DATASET_RUN for the real run. Keep it False for quick staged checks.
OVERFIT_SANITY_RUN = False
FULL_DATASET_RUN   = False
STAGED_SAMPLES     = None if FULL_DATASET_RUN else 100

BATCH_SIZE = 4
LR         = 5e-4   # constant learning rate (Adam)
EPOCHS     = 30 if FULL_DATASET_RUN else 10
GRAD_CLIP  = 10.0   # global gradient-norm clip threshold

# ── Dataset size and augmentation ─────────────────────────────────────────
MAX_TRAIN = 20 if OVERFIT_SANITY_RUN else STAGED_SAMPLES
MAX_VAL   = 20 if OVERFIT_SANITY_RUN else STAGED_SAMPLES
VAL_SPLIT = "train" if OVERFIT_SANITY_RUN else "val"
TRAIN_RANDOM_ROTATION = not OVERFIT_SANITY_RUN
ROTATION_RANGE_DEG = (-20.0, 20.0)
ROTATION_FILL = (128, 128, 128)

if OVERFIT_SANITY_RUN:
    RUN_NAME = "final_overfit"
elif STAGED_SAMPLES is None:
    RUN_NAME = "final_full"
else:
    RUN_NAME = f"final_stage{STAGED_SAMPLES}"
CHECKPOINT_DIR = f"checkpoints_{RUN_NAME}"
LOG_FILE = f"outputs/train_log_{RUN_NAME}.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Anchors and objectness target ──────────────────────────────────────────
# If enabled, ANCHORS is replaced after the training split is loaded.
ESTIMATE_ANCHORS_FROM_DATA = True
ANCHOR_SAMPLE_LIMIT = None
ANCHORS = np.array([[0.08, 0.06],
                    [0.20, 0.15]], dtype=np.float64)

# "iou" is closer to YOLO confidence: objectness learns a detached overlap target.
# Use "binary" if you want the older 1/0 objectness behaviour.
OBJECTNESS_TARGET = "iou"   # choices: "iou", "anchor_iou", "binary"

# ── Loss weights (λ) ──────────────────────────────────────────────────────
# λ_LOC >> 1 because localisation errors dominate early in training.
# λ_THETA is moderate; angle prediction is noisy early in training.
L_LOC   = 5.0
L_OBJ   = 1.0
L_NOOBJ = 0.5
L_CLS   = 1.0
L_THETA = 1.0

# ── Evaluation defaults ───────────────────────────────────────────────────
DEFAULT_CONF_THRESH = 0.3
EVAL_IOU_THRESH = 0.5
CONF_THRESHOLDS = (0.3, 0.5, 0.7, 0.8, 0.9)

# ── VOC class list ─────────────────────────────────────────────────────────
VOC_CLASSES = [
    "aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
    "chair","cow","diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

print(f"Grid {S}x{S} | {B} anchors | {C} classes | img {IMG_SIZE}x{IMG_SIZE}")
print(f"Output tensor shape per image: ({S}, {S}, {B}, {6+C})")
print(f"Run: {RUN_NAME} | train max={MAX_TRAIN} | val split={VAL_SPLIT} max={MAX_VAL}")
print(f"Train random rotation: {TRAIN_RANDOM_ROTATION} range={ROTATION_RANGE_DEG}")
print(f"Objectness target: {OBJECTNESS_TARGET} | estimate anchors: {ESTIMATE_ANCHORS_FROM_DATA}")


## Section 1 — Dataset: VOC Bounding Boxes → Ellipse Annotations

**Conversion rationale.**  
A Pascal VOC bounding box `(xmin, ymin, xmax, ymax)` is the smallest axis-aligned
rectangle containing the object. We *inscribe* an ellipse inside it:

```
cx = (xmin + xmax) / 2 / W      # normalised horizontal centre
cy = (ymin + ymax) / 2 / H      # normalised vertical centre
a  = (xmax - xmin) / 2 / W      # semi-major axis (horizontal half-width)
b  = (ymax - ymin) / 2 / H      # semi-minor axis (vertical half-height)
θ  = 0                          # axis-aligned; rotation comes from augmentation
```

This is the standard *bounding-ellipse* conversion used when ground-truth masks are
unavailable. For objects that are truly elongated (aeroplane, boat, train) the ellipse
is a much tighter fit than the rectangle.

During training, the image is resized first and then optionally rotated once per sample access. The rotation uses gray border fill and updates each ellipse center plus `theta`, so rotated training samples provide nonzero orientation targets. Validation samples are not randomly rotated.


In [ ]:
def find_voc_root(search_root):
    """Walk the directory tree to locate the Pascal VOC root folder."""
    needed = ("JPEGImages", "Annotations", "ImageSets")
    if all(os.path.isdir(os.path.join(search_root, d)) for d in needed):
        return search_root
    for root, dirs, _ in os.walk(search_root):
        if all(d in dirs for d in needed):
            return root
    raise FileNotFoundError(
        f"VOC root not found under {search_root!r}. "
        "Expected JPEGImages/, Annotations/, ImageSets/."
    )


def normalise_theta(theta):
    """Ellipse orientation is pi-periodic, so keep theta in [0, pi)."""
    return float(theta % np.pi)


def rotate_normalized_point(cx, cy, angle_deg):
    """Rotate a normalized point around the image center."""
    angle = np.deg2rad(angle_deg)
    dx, dy = cx - 0.5, cy - 0.5
    cos_t, sin_t = np.cos(angle), np.sin(angle)
    return float(0.5 + cos_t * dx - sin_t * dy), float(0.5 + sin_t * dx + cos_t * dy)


def rotate_sample(img, targets, angle_deg, fill=ROTATION_FILL):
    """Rotate a resized PIL image once and update ellipse centers/theta."""
    rotated_img = img.rotate(
        -angle_deg,                      # PIL positive is counter-clockwise
        resample=Image.BILINEAR,
        expand=False,
        fillcolor=fill,
    )
    theta_delta = np.deg2rad(angle_deg)
    rotated_targets = []
    for target in targets:
        cx, cy = rotate_normalized_point(target["cx"], target["cy"], angle_deg)
        if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
            continue
        new_target = dict(target)
        new_target["cx"] = cx
        new_target["cy"] = cy
        new_target["theta"] = normalise_theta(target["theta"] + theta_delta)
        rotated_targets.append(new_target)
    return rotated_img, rotated_targets


def parse_voc_xml(xml_path):
    """
    Parse a single VOC XML annotation file.
    Returns dict with 'file' (image filename) and 'targets' (list of ellipse dicts).
    Skips objects marked as 'difficult' and any degenerate boxes.
    """
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W, H = int(size.findtext("width")), int(size.findtext("height"))
    targets = []
    for obj in root.iter("object"):
        name = obj.findtext("name", "")
        if name not in CLASS_TO_IDX:
            continue
        if int(obj.findtext("difficult", "0")):
            continue
        b = obj.find("bndbox")
        xmin = max(0.0, float(b.findtext("xmin")))
        ymin = max(0.0, float(b.findtext("ymin")))
        xmax = min(W,   float(b.findtext("xmax")))
        ymax = min(H,   float(b.findtext("ymax")))
        if xmax <= xmin or ymax <= ymin:
            continue
        targets.append({
            "class_id": CLASS_TO_IDX[name],
            "cx":   (xmin + xmax) / 2 / W,
            "cy":   (ymin + ymax) / 2 / H,
            "a":    (xmax - xmin) / 2 / W,
            "b":    (ymax - ymin) / 2 / H,
            "theta": 0.0,
        })
    return {"file": root.findtext("filename"), "targets": targets}


def load_split(voc_root, split, max_n=None):
    """Load all (or up to max_n) annotated samples for the given VOC split."""
    split_file = os.path.join(voc_root, "ImageSets", "Main", f"{split}.txt")
    ids = [line.strip().split()[0] for line in open(split_file) if line.strip()]
    ann_dir = os.path.join(voc_root, "Annotations")
    samples = []
    for img_id in ids:
        xml = os.path.join(ann_dir, f"{img_id}.xml")
        if not os.path.exists(xml):
            continue
        s = parse_voc_xml(xml)
        if s["targets"]:
            samples.append(s)
        if max_n is not None and len(samples) >= max_n:
            break
    return samples


class VOCEllipseDataset:
    """
    Minimal PyTorch-free dataset.
    Returns (img_array, targets):
      img_array : float32 CHW in [0, 1], shape (3, IMG_SIZE, IMG_SIZE)
      targets   : list of dicts {class_id, cx, cy, a, b, theta}

    Images are resized first. If random_rotation=True, each __getitem__ call
    applies one random rotation and updates ellipse center/theta targets.
    """
    def __init__(self, voc_root, split, max_n=None, random_rotation=False,
                 rotation_range=ROTATION_RANGE_DEG):
        self.img_dir = os.path.join(voc_root, "JPEGImages")
        self.samples = load_split(voc_root, split, max_n)
        self.random_rotation = bool(random_rotation)
        self.rotation_range = rotation_range

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = (Image.open(os.path.join(self.img_dir, s["file"]))
                     .convert("RGB")
                     .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        targets = [dict(t) for t in s["targets"]]
        if self.random_rotation:
            angle = float(np.random.uniform(*self.rotation_range))
            img, targets = rotate_sample(img, targets, angle)
        arr = np.array(img, dtype=np.float32).transpose(2, 0, 1) / 255.0
        return arr, targets



def axis_shape_iou(axis, anchors):
    """Fast shape IoU between one ellipse axis pair and B anchor axis pairs."""
    a, b = axis
    inter = np.pi * np.minimum(a, anchors[:, 0]) * np.minimum(b, anchors[:, 1])
    union = np.pi * (a * b + anchors[:, 0] * anchors[:, 1]
                     - np.minimum(a, anchors[:, 0]) * np.minimum(b, anchors[:, 1]))
    return inter / np.maximum(union, 1e-9)


def estimate_anchors_from_samples(samples, B, max_targets=None, n_iter=40):
    """K-means style anchor estimation using ellipse semi-axes (a, b)."""
    axes = []
    for sample in samples:
        for target in sample["targets"]:
            if target["a"] > 0 and target["b"] > 0:
                axes.append([target["a"], target["b"]])
    axes = np.array(axes, dtype=np.float64)
    if len(axes) == 0:
        return ANCHORS.copy()

    if max_targets is not None and len(axes) > max_targets:
        rng = np.random.default_rng(42)
        axes = axes[rng.choice(len(axes), size=max_targets, replace=False)]

    # Initialise anchors from object-area quantiles so small and larger objects are represented.
    order = np.argsort(axes[:, 0] * axes[:, 1])
    quantiles = np.linspace(0.15, 0.85, B)
    init_idx = [order[min(int(q * (len(order) - 1)), len(order) - 1)] for q in quantiles]
    anchors = axes[init_idx].copy()

    for _ in range(n_iter):
        ious = np.stack([axis_shape_iou(axis, anchors) for axis in axes], axis=0)
        assign = ious.argmax(axis=1)
        new_anchors = anchors.copy()
        for bi in range(B):
            members = axes[assign == bi]
            if len(members):
                new_anchors[bi] = np.median(members, axis=0)
        if np.allclose(new_anchors, anchors, rtol=1e-4, atol=1e-5):
            break
        anchors = new_anchors

    anchors = anchors[np.argsort(anchors[:, 0] * anchors[:, 1])]
    return anchors.astype(np.float64)


# ── Build train / val datasets ─────────────────────────────────────────────
voc_root = find_voc_root(VOC_SEARCH_ROOT)
train_ds = VOCEllipseDataset(
    voc_root, "train", MAX_TRAIN,
    random_rotation=TRAIN_RANDOM_ROTATION,
    rotation_range=ROTATION_RANGE_DEG,
)
val_ds   = VOCEllipseDataset(voc_root, VAL_SPLIT, MAX_VAL, random_rotation=False)

if ESTIMATE_ANCHORS_FROM_DATA:
    ANCHORS = estimate_anchors_from_samples(
        train_ds.samples, B, max_targets=ANCHOR_SAMPLE_LIMIT)

print(f"VOC root : {voc_root}")
print(f"Train    : {len(train_ds)} samples")
print(f"Val      : {len(val_ds)}   samples from split={VAL_SPLIT}")
print(f"Rotation : train random={TRAIN_RANDOM_ROTATION}, range={ROTATION_RANGE_DEG}")
print(f"Anchors  : {np.round(ANCHORS, 4).tolist()}")
print(f"Example  : {train_ds[0][1][0]}")


In [ ]:
def visualize_gt(dataset, n=6, save="figures/gt_ellipses.png"):
    """Draw ground-truth ellipses over resized training images."""
    cols, rows = 3, (n + 2) // 3
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, range(min(n, len(dataset)))):
        img, targets = dataset[i]
        ax.imshow(img.transpose(1, 2, 0).clip(0, 1))
        for t in targets:
            ax.add_patch(MplEllipse(
                (t["cx"] * IMG_SIZE, t["cy"] * IMG_SIZE),
                2 * t["a"] * IMG_SIZE, 2 * t["b"] * IMG_SIZE,
                angle=np.rad2deg(t["theta"]),
                fill=False, edgecolor="lime", lw=1.5,
            ))
            ax.text(t["cx"] * IMG_SIZE, t["cy"] * IMG_SIZE,
                    VOC_CLASSES[t["class_id"]], color="white", fontsize=7,
                    ha="center", va="center")
    fig.suptitle("Ground-truth ellipses (lime) on resized VOC images")
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

visualize_gt(train_ds)


In [ ]:
def visualize_rotation_preview(dataset, n=3, angles=(-20.0, 0.0, 20.0), save="figures/rotation_preview.png"):
    """Show how image rotation changes ellipse center/theta targets."""
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, len(angles), figsize=(4 * len(angles), 4 * n))
    axes = np.array(axes).reshape(n, len(angles))

    for row in range(n):
        sample = dataset.samples[row]
        img = (Image.open(os.path.join(dataset.img_dir, sample["file"]))
                    .convert("RGB")
                    .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        targets = [dict(t) for t in sample["targets"]]

        for col, angle in enumerate(angles):
            ax = axes[row, col]
            if abs(angle) > 1e-9:
                shown_img, shown_targets = rotate_sample(img, targets, angle)
            else:
                shown_img, shown_targets = img, targets
            arr = np.array(shown_img, dtype=np.float32) / 255.0
            ax.imshow(arr)
            ax.set_title(f"rotation {angle:+.0f} deg")
            ax.axis("off")
            for t in shown_targets:
                ax.add_patch(MplEllipse(
                    (t["cx"] * IMG_SIZE, t["cy"] * IMG_SIZE),
                    2 * t["a"] * IMG_SIZE, 2 * t["b"] * IMG_SIZE,
                    angle=np.rad2deg(t["theta"]),
                    fill=False, edgecolor="lime", lw=1.4,
                ))

    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

visualize_rotation_preview(train_ds)


## Section 2 — NumPy Neural Network Layers (Manual Backprop)

Every layer exposes:
- `forward(x, training)` — compute output and cache values needed for backprop
- `backward(dout)` — return gradient w.r.t. input; accumulate `dW`, `db` in-place
- `params()` — list of `(param, grad)` pairs handed to the optimiser

### Convolution implementation
Forward: `im2col` reshapes input patches into a matrix, then a single `@` replaces
all nested loops — same result, ~100× faster.  
Backward: `dW = X_cols^T @ dY_flat`, `dX = dY_flat @ W_flat` then `col2im` scatters
gradients back.


In [ ]:
# ── im2col / col2im ────────────────────────────────────────────────────────
# im2col uses numpy stride tricks to create a VIEW of the input where each
# column corresponds to one receptive field. No data is copied until .copy().

def im2col(x, kH, kW, stride=1, pad=0):
    """
    Extract sliding patches from x and lay them out as matrix rows.
    Returns (col, oH, oW) where col has shape (N*oH*oW, C*kH*kW).
    Uses stride tricks: O(1) overhead — no Python loops.
    """
    N, C, H, W = x.shape
    if pad:
        x = np.pad(x, ((0,0),(0,0),(pad,pad),(pad,pad)), mode="constant")
    _, _, Hp, Wp = x.shape
    oH = (Hp - kH) // stride + 1
    oW = (Wp - kW) // stride + 1
    shape   = (N, C, kH, kW, oH, oW)
    strides = (
        x.strides[0], x.strides[1],
        x.strides[2], x.strides[3],
        x.strides[2] * stride, x.strides[3] * stride,
    )
    col = np.lib.stride_tricks.as_strided(x, shape=shape, strides=strides)
    # Transpose to (N, oH, oW, C, kH, kW) then flatten spatial dims and kernel dims
    return col.transpose(0, 4, 5, 1, 2, 3).copy().reshape(N * oH * oW, -1), oH, oW


def col2im(dcol, x_shape, kH, kW, stride=1, pad=0):
    """
    Scatter patch gradients back into input-gradient tensor.
    Accumulates overlapping contributions (correct for stride < kernel size).
    """
    N, C, H, W = x_shape
    Hp, Wp = H + 2 * pad, W + 2 * pad
    oH = (Hp - kH) // stride + 1
    oW = (Wp - kW) // stride + 1
    dx_pad = np.zeros((N, C, Hp, Wp))
    # dcol: (N*oH*oW, C*kH*kW) → (N, oH, oW, C, kH, kW)
    dcol = dcol.reshape(N, oH, oW, C, kH, kW)
    for r in range(kH):
        for c in range(kW):
            # accumulate into the correct stride-offset positions
            dx_pad[:, :, r:r + oH * stride:stride, c:c + oW * stride:stride] += (
                dcol[:, :, :, :, r, c].transpose(0, 3, 1, 2)
            )
    return dx_pad[:, :, pad:pad + H, pad:pad + W] if pad else dx_pad


In [ ]:
# ── Conv2D ─────────────────────────────────────────────────────────────────
class Conv2D:
    """
    2-D cross-correlation layer.
    Weight W shape: (out_ch, in_ch, kH, kW), initialised with He normal.

    Backward derivation (chain rule for Y = X * K + b):
        dW = X_cols^T @ dY_flat     (accumulate filter gradients)
        db = sum(dY_flat, axis=0)
        dX = col2im(dY_flat @ W_flat)   (scatter back to input space)
    """
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, pad=None):
        self.stride = stride
        self.pad    = kernel // 2 if pad is None else pad
        self.k      = kernel
        fan_in = in_ch * kernel * kernel
        self.W  = np.random.randn(out_ch, in_ch, kernel, kernel) * np.sqrt(2.0 / fan_in)
        self.b  = np.zeros(out_ch)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._cache = None

    def forward(self, x, training=True):
        N = x.shape[0]
        col, oH, oW = im2col(x, self.k, self.k, self.stride, self.pad)
        W_flat = self.W.reshape(self.W.shape[0], -1)          # (out_ch, C*k*k)
        out = (col @ W_flat.T + self.b).reshape(N, oH, oW, -1).transpose(0, 3, 1, 2)
        if training:
            self._cache = (x, col, oH, oW)
        return out

    def backward(self, dout):
        x, col, oH, oW = self._cache
        N = x.shape[0]
        # dout: (N, out_ch, oH, oW) → flatten to (N*oH*oW, out_ch)
        g = dout.transpose(0, 2, 3, 1).reshape(-1, self.W.shape[0])
        self.dW[...] = (g.T @ col).reshape(self.W.shape)
        self.db[...] = g.sum(axis=0)
        W_flat = self.W.reshape(self.W.shape[0], -1)
        dx = col2im(g @ W_flat, x.shape, self.k, self.k, self.stride, self.pad)
        return dx

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]


class Conv1x1(Conv2D):
    """
    Pointwise (1×1) convolution — cross-channel mixing / dimensionality reduction.
    No spatial context; equivalent to a per-pixel Dense layer applied across channels.
    Inherits Conv2D with kernel=1, pad=0.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__(in_ch, out_ch, kernel=1, stride=1, pad=0)


In [ ]:
# ── MaxPool2D ───────────────────────────────────────────────────────────────
class MaxPool2D:
    """
    Non-overlapping 2-D max-pool (pool == stride).
    Uses numpy reshape instead of Python loops — no im2col needed.

    Forward:  reshape x to (N,C,oH,p,oW,p), take max over pool dims (3,5).
    Backward: route gradient to the winning position; split ties equally.
    """
    def __init__(self, pool=2):
        self.pool   = pool
        self._cache = None

    def forward(self, x, training=True):
        N, C, H, W = x.shape
        p   = self.pool
        oH, oW = H // p, W // p
        # Crop to exact multiple then reshape into (N, C, oH, p, oW, p)
        patches = x[:, :, :oH * p, :oW * p].reshape(N, C, oH, p, oW, p)
        out = patches.max(axis=(3, 5))          # (N, C, oH, oW)
        if training:
            # Build a fractional mask so ties share the gradient equally
            mask = (patches == out[:, :, :, None, :, None]).astype(np.float32)
            mask /= mask.sum(axis=(3, 5), keepdims=True).clip(min=1)
            self._cache = (x.shape, mask, oH, oW)
        return out

    def backward(self, dout):
        x_shape, mask, oH, oW = self._cache
        N, C, H, W = x_shape
        p = self.pool
        # Broadcast dout over pool dims then reshape back to input size
        dx_patches = dout[:, :, :, None, :, None] * mask  # (N,C,oH,p,oW,p)
        dx = np.zeros(x_shape)
        dx[:, :, :oH * p, :oW * p] = dx_patches.reshape(N, C, oH * p, oW * p)
        return dx

    def params(self):
        return []


# ── Activation ──────────────────────────────────────────────────────────────
class LeakyReLU:
    """
    f(x) = x  if x > 0  else  alpha * x
    Standard YOLO choice: alpha = 0.1 keeps negative activations alive.
    Backward: f'(x) = 1 if x > 0 else alpha  (element-wise chain rule).
    """
    def __init__(self, alpha=0.1):
        self.alpha  = alpha
        self._cache = None

    def forward(self, x, training=True):
        if training:
            self._cache = x
        return np.where(x > 0, x, self.alpha * x)

    def backward(self, dout):
        return dout * np.where(self._cache > 0, 1.0, self.alpha)

    def params(self):
        return []


# ── Dense (fully-connected) ─────────────────────────────────────────────────
class Dense:
    """
    Z = X @ W + b

    Backward (from the project guide):
        dW = X^T @ dZ
        db = sum(dZ, axis=0)
        dX = dZ @ W^T
    """
    def __init__(self, in_dim, out_dim):
        self.W  = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b  = np.zeros(out_dim)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._cache = None

    def forward(self, x, training=True):
        if training:
            self._cache = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW[...] = self._cache.T @ dout
        self.db[...] = dout.sum(axis=0)
        return dout @ self.W.T

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]


## Section 3 — Adam Optimiser (from scratch)

Implements the update rule from Kingma & Ba (2015):

```
m_t = β1·m_{t-1} + (1−β1)·g_t            # first moment (mean)
v_t = β2·v_{t-1} + (1−β2)·g_t²           # second moment (uncentred variance)
m̂_t = m_t / (1−β1^t)                     # bias correction
v̂_t = v_t / (1−β2^t)
θ_t  = θ_{t-1} − α · m̂_t / (√v̂_t + ε)  # parameter update
```


In [ ]:
class Adam:
    """
    Adam optimiser operating directly on (param, grad) array pairs.
    Typical settings: lr=1e-3, beta1=0.9, beta2=0.999.

    Usage:
        opt = Adam(model.params(), lr=LR)
        ...
        opt.zero_grad()      # zero all grad arrays in-place
        model.backward(grad) # fill grad arrays via backprop
        opt.step()           # update params in-place
    """
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params            # list of (param_arr, grad_arr)
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.t = 0
        self.m = [np.zeros_like(p) for p, _ in params]   # first moments
        self.v = [np.zeros_like(p) for p, _ in params]   # second moments

    def step(self):
        self.t += 1
        bc1 = 1.0 - self.beta1 ** self.t   # bias correction denominators
        bc2 = 1.0 - self.beta2 ** self.t
        for i, (p, g) in enumerate(self.params):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for _, g in self.params:
            g[:] = 0.0


## Section 4 — EllipseNet Architecture

```
Input  (N, 3, 224, 224)
  │
  ├─ Conv(3→16,  3×3) + LeakyReLU  + MaxPool(2) →  (N, 16, 64, 64)
  ├─ Conv1×1(16→8)    + LeakyReLU               →  (N,  8, 64, 64)  ← channel bottleneck
  ├─ Conv(8→32,  3×3) + LeakyReLU  + MaxPool(2) →  (N, 32, 32, 32)
  ├─ Conv(32→64, 3×3) + LeakyReLU  + MaxPool(2) →  (N, 64, 16, 16)
  ├─ Conv1×1(64→32)   + LeakyReLU               →  (N, 32, 16, 16)  ← channel bottleneck
  ├─ Conv(32→64, 3×3) + LeakyReLU  + MaxPool(2) →  (N, 64,  8,  8)  = S×S feature map
  │
  ├─ Flatten → (N, 64·7·7 = 3136)
  ├─ Dense(3136 → 512) + LeakyReLU
  └─ Dense(512  → S·S·B·(6+C))

Output reshape → (N, S, S, B, 6+C)
```

**Per-anchor output layout** `[conf_raw | cx_raw | cy_raw | log_a | log_b | θ_raw | cls×C]`

- `conf_raw` → sigmoid → objectness probability  
- `cx_raw, cy_raw` → sigmoid → fractional offset within cell → decode to absolute cx, cy  
- `log_a, log_b` → exp → scale factor on anchor semi-axes  
- `θ_raw` → π·sigmoid → orientation in [0, π)  
- `cls[0..C-1]` → softmax → class probabilities

**Architecture satisfies all constraints:** 4+ Conv layers, 2× Conv1×1, MaxPool at every stage, LeakyReLU activations throughout.


In [ ]:
class EllipseNet:
    """
    YOLO-style ellipse detector built entirely from NumPy layers.

    The backbone reduces 224×224 → 7×7 feature maps through five /2 stages.
    A two-layer FC head reshapes these into the (S, S, B, 6+C) output grid.
    """

    def __init__(self, S=7, B=2, C=20, anchors=None):
        self.S       = S
        self.B       = B
        self.C       = C
        self.anchors = np.array(ANCHORS if anchors is None else anchors)
        out_per_cell = B * (6 + C)

        # ── Backbone: 5 Conv/Pool stages + 2 Conv1×1 bottlenecks ─────────
        self.backbone = [
            # Block 1: 224 → 112
            Conv2D(3,   16, 3), LeakyReLU(), MaxPool2D(2),
            # Bottleneck 1: compress channels
            Conv1x1(16,  8),   LeakyReLU(),
            # Block 2: 112 → 56
            Conv2D(8,   32, 3), LeakyReLU(), MaxPool2D(2),
            # Block 3: 56 → 28
            Conv2D(32,  64, 3), LeakyReLU(), MaxPool2D(2),
            # Bottleneck 2: compress channels
            Conv1x1(64, 32),   LeakyReLU(),
            # Block 4: 28 → 14
            Conv2D(32,  64, 3), LeakyReLU(), MaxPool2D(2),
            # Block 5: 14 → 7 (= S)
            Conv2D(64,  64, 3), LeakyReLU(), MaxPool2D(2),
        ]

        # ── Detection head: flatten → Dense → reshape ─────────────────────
        fc_in = 64 * S * S      # 64 channels × 7×7 spatial = 3136
        self.fc1    = Dense(fc_in, 512)
        self.fc_act = LeakyReLU()
        self.fc2    = Dense(512, S * S * out_per_cell)
        self.head   = [self.fc1, self.fc_act, self.fc2]

    # ── Forward pass ─────────────────────────────────────────────────────
    def forward(self, x, training=True):
        """x: (N, 3, H, W) → raw output (N, S, S, B, 6+C)."""
        for layer in self.backbone:
            x = layer.forward(x, training)
        N = x.shape[0]
        self._feat_shape = x.shape      # cache for backward reshape
        x = x.reshape(N, -1)
        for layer in self.head:
            x = layer.forward(x, training)
        return x.reshape(N, self.S, self.S, self.B, 6 + self.C)

    # ── Backward pass ─────────────────────────────────────────────────────
    def backward(self, dout):
        """dout: (N, S, S, B, 6+C) — upstream gradient from loss."""
        N = dout.shape[0]
        g = dout.reshape(N, -1)
        for layer in reversed(self.head):
            g = layer.backward(g)
        g = g.reshape(self._feat_shape)
        for layer in reversed(self.backbone):
            g = layer.backward(g)

    # ── Parameter list ────────────────────────────────────────────────────
    def params(self):
        p = []
        for layer in self.backbone + self.head:
            p.extend(layer.params())
        return p

    # ── Save / load weights ───────────────────────────────────────────────
    def save(self, path):
        np.savez(path, **{f"p{i}": p for i, (p, _) in enumerate(self.params())})
        print(f"Checkpoint saved → {path}")

    def load(self, path):
        data = np.load(path)
        for i, (p, _) in enumerate(self.params()):
            p[:] = data[f"p{i}"]
        print(f"Checkpoint loaded ← {path}")


# ── Instantiate and run a shape-check forward pass ────────────────────────
model = EllipseNet(S=S, B=B, C=C)
x_dummy = np.zeros((2, 3, IMG_SIZE, IMG_SIZE), np.float32)
out_dummy = model.forward(x_dummy, training=False)

print("Input  shape:", x_dummy.shape)
print("Output shape:", out_dummy.shape, "  expected:", (2, S, S, B, 6 + C))
assert out_dummy.shape == (2, S, S, B, 6 + C), "Shape mismatch!"

total_params = sum(p.size for p, _ in model.params())
print(f"Total parameters: {total_params:,}")
print("Shape check PASSED")


## Section 5 — Ellipse IoU & Loss Function

### Ellipse IoU (Monte Carlo)
Exact analytic intersection of two oriented ellipses has no closed form.
We estimate it via **Monte Carlo sampling**: draw `n` uniform random points inside
the union bounding box of the two ellipses, then count what fraction fall inside
the intersection vs. the union.

A point `(px, py)` is inside ellipse `(cx, cy, a, b, θ)` when:
```
x' = (px−cx)·cos θ + (py−cy)·sin θ
y' =−(px−cx)·sin θ + (py−cy)·cos θ
(x'/a)² + (y'/b)² ≤ 1
```

### Loss components
| Term | Formula | Mask |
|------|---------|------|
| Objectness (obj) | BCE(σ(conf), 1) | responsible anchors |
| Objectness (noobj) | BCE(σ(conf), 0) | all other anchors |
| Centre | MSE(σ(cx_raw)−gi/S, cx_gt) | obj |
| Size (log space) | MSE(log_a_raw, log(a_gt/a_anc)) | obj |
| Angle (π-periodic) | mean min(\|Δθ\|, π−\|Δθ\|) | obj |
| Classification | Softmax-CE(logits, class_id) | obj |


In [ ]:
# ── Utility activations ────────────────────────────────────────────────────

def sigmoid(x):
    """Numerically stable sigmoid: clips to avoid exp overflow."""
    return np.where(x >= 0,
                    1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1.0 - s)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


# ── Ellipse IoU via Monte Carlo ─────────────────────────────────────────────
def ellipse_iou_mc(e1, e2, n=400):
    """
    Estimate IoU between two ellipses e1=(cx,cy,a,b,theta) and e2.
    Spatial coordinates are normalised to [0,1]. Used for evaluation/NMS.
    """
    cx1, cy1, a1, b1, t1 = e1
    cx2, cy2, a2, b2, t2 = e2
    a1, b1 = max(a1, 1e-6), max(b1, 1e-6)
    a2, b2 = max(a2, 1e-6), max(b2, 1e-6)

    def aabb(cx, cy, a, b, t):
        tx = np.sqrt(a**2 * np.cos(t)**2 + b**2 * np.sin(t)**2)
        ty = np.sqrt(a**2 * np.sin(t)**2 + b**2 * np.cos(t)**2)
        return cx - tx, cy - ty, cx + tx, cy + ty

    def inside(px, py, cx, cy, a, b, t):
        dx, dy = px - cx, py - cy
        xr =  dx * np.cos(t) + dy * np.sin(t)
        yr = -dx * np.sin(t) + dy * np.cos(t)
        return (xr / a)**2 + (yr / b)**2 <= 1.0

    x0, y0, x1, y1 = aabb(cx1, cy1, a1, b1, t1)
    x2, y2, x3, y3 = aabb(cx2, cy2, a2, b2, t2)
    ux0, uy0 = min(x0, x2), min(y0, y2)
    ux1, uy1 = max(x1, x3), max(y1, y3)
    if ux1 <= ux0 or uy1 <= uy0:
        return 0.0

    px = np.random.uniform(ux0, ux1, n)
    py = np.random.uniform(uy0, uy1, n)
    i1 = inside(px, py, cx1, cy1, a1, b1, t1)
    i2 = inside(px, py, cx2, cy2, a2, b2, t2)
    union = (i1 | i2).sum()
    return 0.0 if union == 0 else float((i1 & i2).sum()) / float(union)


def ellipse_aabb_iou(cx1, cy1, a1, b1, t1, cx2, cy2, a2, b2, t2):
    """Fast deterministic IoU proxy: overlap of ellipse bounding boxes."""
    def bounds(cx, cy, a, b, t):
        tx = np.sqrt(a**2 * np.cos(t)**2 + b**2 * np.sin(t)**2)
        ty = np.sqrt(a**2 * np.sin(t)**2 + b**2 * np.cos(t)**2)
        return cx - tx, cy - ty, cx + tx, cy + ty

    x1a, y1a, x1b, y1b = bounds(cx1, cy1, a1, b1, t1)
    x2a, y2a, x2b, y2b = bounds(cx2, cy2, a2, b2, t2)
    ix0, iy0 = max(x1a, x2a), max(y1a, y2a)
    ix1, iy1 = min(x1b, x2b), min(y1b, y2b)
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    area1 = max(0.0, x1b - x1a) * max(0.0, y1b - y1a)
    area2 = max(0.0, x2b - x2a) * max(0.0, y2b - y2a)
    return inter / max(area1 + area2 - inter, 1e-9)


# ── Target assignment ───────────────────────────────────────────────────────
def build_targets(targets_batch, S, B, anchors):
    """
    Assign each ground-truth ellipse to the best anchor in the correct grid cell.
    The returned t_anchor_iou is a static shape-overlap target useful for diagnostics
    or the cheaper anchor-IoU objectness mode.
    """
    N = len(targets_batch)
    obj   = np.zeros((N, S, S, B), dtype=bool)
    noobj = np.ones ((N, S, S, B), dtype=bool)
    t_cx  = np.zeros((N, S, S, B))
    t_cy  = np.zeros((N, S, S, B))
    t_a   = np.zeros((N, S, S, B))
    t_b   = np.zeros((N, S, S, B))
    t_th  = np.zeros((N, S, S, B))
    t_cls = np.zeros((N, S, S, B), dtype=int)
    t_anchor_iou = np.zeros((N, S, S, B))

    for ni, gts in enumerate(targets_batch):
        for gt in gts:
            cx, cy = gt["cx"], gt["cy"]
            a,  b  = gt["a"],  gt["b"]
            theta  = gt["theta"]
            cls    = int(gt["class_id"])

            gi = min(int(cx * S), S - 1)
            gj = min(int(cy * S), S - 1)

            best_bi, best_iou = 0, -1.0
            for bi in range(B):
                an_a, an_b = anchors[bi]
                inter = np.pi * min(a, an_a) * min(b, an_b)
                union = np.pi * (a * b + an_a * an_b - min(a, an_a) * min(b, an_b))
                iou   = inter / max(union, 1e-9)
                if iou > best_iou:
                    best_iou, best_bi = iou, bi

            obj  [ni, gj, gi, best_bi] = True
            noobj[ni, gj, gi, best_bi] = False
            t_cx [ni, gj, gi, best_bi] = cx * S - gi
            t_cy [ni, gj, gi, best_bi] = cy * S - gj
            t_a  [ni, gj, gi, best_bi] = np.log(max(a / anchors[best_bi, 0], 1e-6))
            t_b  [ni, gj, gi, best_bi] = np.log(max(b / anchors[best_bi, 1], 1e-6))
            t_th [ni, gj, gi, best_bi] = theta
            t_cls[ni, gj, gi, best_bi] = cls
            t_anchor_iou[ni, gj, gi, best_bi] = best_iou

    return obj, noobj, t_cx, t_cy, t_a, t_b, t_th, t_cls, t_anchor_iou


def objectness_iou_targets(raw, obj, t_cx, t_cy, t_a, t_b, t_th, anchors, S):
    """Detached YOLO-style confidence target: predicted-vs-GT AABB IoU."""
    targets = np.zeros(raw.shape[:4], dtype=np.float64)
    if not obj.any():
        return targets

    grid = np.arange(S)
    gx, gy = np.meshgrid(grid, grid)
    pred_cx = (sigmoid(raw[..., 1]) + gx[None, :, :, None]) / S
    pred_cy = (sigmoid(raw[..., 2]) + gy[None, :, :, None]) / S
    pred_a  = np.exp(np.clip(raw[..., 3], -6, 2)) * anchors[:, 0]
    pred_b  = np.exp(np.clip(raw[..., 4], -6, 2)) * anchors[:, 1]
    pred_th = np.pi * sigmoid(raw[..., 5])

    obj_idx = np.argwhere(obj)
    for ni, gj, gi, bi in obj_idx:
        gt_cx = (t_cx[ni, gj, gi, bi] + gi) / S
        gt_cy = (t_cy[ni, gj, gi, bi] + gj) / S
        gt_a  = np.exp(t_a[ni, gj, gi, bi]) * anchors[bi, 0]
        gt_b  = np.exp(t_b[ni, gj, gi, bi]) * anchors[bi, 1]
        gt_th = t_th[ni, gj, gi, bi]
        targets[ni, gj, gi, bi] = ellipse_aabb_iou(
            pred_cx[ni, gj, gi, bi], pred_cy[ni, gj, gi, bi],
            pred_a[ni, gj, gi, bi], pred_b[ni, gj, gi, bi], pred_th[ni, gj, gi, bi],
            gt_cx, gt_cy, gt_a, gt_b, gt_th)
    return targets


# ── Combined ellipse detection loss ─────────────────────────────────────────
def ellipse_loss(raw, targets_batch, anchors, S, B, C,
                 l_loc=5.0, l_obj=1.0, l_noobj=0.5,
                 l_cls=1.0, l_theta=1.0,
                 objectness_target="iou"):
    """
    Compute the YOLO-style ellipse loss and its gradient w.r.t. raw.

    Objectness modes:
      binary     : old behaviour, object target is 1.0
      anchor_iou : object target is the assigned anchor's shape IoU
      iou        : object target is a detached predicted-vs-GT AABB IoU proxy
    """
    grad = np.zeros_like(raw)
    obj, noobj, t_cx, t_cy, t_a, t_b, t_th, t_cls, t_anchor_iou = build_targets(
        targets_batch, S, B, anchors)

    # ── Objectness ─────────────────────────────────────────────────────────
    conf     = sigmoid(raw[..., 0])
    n_obj    = max(obj.sum(),   1)
    n_noobj  = max(noobj.sum(), 1)
    eps      = 1e-7

    conf_target = np.zeros_like(conf)
    if obj.any():
        if objectness_target == "binary":
            conf_target[obj] = 1.0
        elif objectness_target == "anchor_iou":
            conf_target[obj] = t_anchor_iou[obj]
        elif objectness_target == "iou":
            conf_target[obj] = objectness_iou_targets(
                raw, obj, t_cx, t_cy, t_a, t_b, t_th, anchors, S)[obj]
        else:
            raise ValueError("objectness_target must be 'iou', 'anchor_iou', or 'binary'")

    g_conf = np.zeros_like(conf)
    loss_obj = loss_noobj = 0.0

    if obj.any():
        c = conf[obj].clip(eps, 1 - eps)
        y = conf_target[obj].clip(0.0, 1.0)
        loss_obj = (-(y * np.log(c) + (1.0 - y) * np.log(1.0 - c))).mean()
        g_conf[obj] = (conf[obj] - y) / n_obj

    if noobj.any():
        c = conf[noobj].clip(eps, 1 - eps)
        loss_noobj = (-(np.log(1.0 - c))).mean()
        g_conf[noobj] = conf[noobj] / n_noobj

    grad[..., 0] = l_obj * g_conf * obj.astype(float) + l_noobj * g_conf * noobj.astype(float)

    if not obj.any():
        total = l_obj * loss_obj + l_noobj * loss_noobj
        return total, grad, dict(total=float(total), loc=0.0, obj=float(loss_obj),
                                  noobj=float(loss_noobj), cls=0.0, theta=0.0)

    # ── Centre (cx, cy): MSE(sigmoid(raw), target) ────────────────────────
    cx_s = sigmoid(raw[..., 1]); cy_s = sigmoid(raw[..., 2])
    ecx = cx_s[obj] - t_cx[obj]; ecy = cy_s[obj] - t_cy[obj]
    loss_cx = (ecx**2).mean(); loss_cy = (ecy**2).mean()
    g_cx = np.zeros_like(cx_s); g_cy = np.zeros_like(cy_s)
    g_cx[obj] = 2 * ecx / n_obj * sigmoid_grad(raw[..., 1][obj])
    g_cy[obj] = 2 * ecy / n_obj * sigmoid_grad(raw[..., 2][obj])
    grad[..., 1] = g_cx * l_loc
    grad[..., 2] = g_cy * l_loc

    # ── Semi-axes: log-anchor loss plus YOLO-style sqrt-axis loss ─────────
    raw_a = raw[..., 3]
    raw_b = raw[..., 4]
    ea = raw_a[obj] - t_a[obj]
    eb = raw_b[obj] - t_b[obj]
    loss_log_ab = 0.5 * (ea**2).mean() + 0.5 * (eb**2).mean()

    pred_a = np.exp(np.clip(raw_a, -6, 2)) * anchors[:, 0]
    pred_b = np.exp(np.clip(raw_b, -6, 2)) * anchors[:, 1]
    tgt_a  = np.exp(t_a) * anchors[:, 0]
    tgt_b  = np.exp(t_b) * anchors[:, 1]

    sqrt_pa = np.sqrt(np.maximum(pred_a[obj], 1e-8))
    sqrt_pb = np.sqrt(np.maximum(pred_b[obj], 1e-8))
    sqrt_ta = np.sqrt(np.maximum(tgt_a[obj], 1e-8))
    sqrt_tb = np.sqrt(np.maximum(tgt_b[obj], 1e-8))
    esa = sqrt_pa - sqrt_ta
    esb = sqrt_pb - sqrt_tb
    loss_sqrt_ab = 0.5 * (esa**2).mean() + 0.5 * (esb**2).mean()

    g_a = np.zeros_like(raw_a); g_b = np.zeros_like(raw_b)
    g_a_obj = 2 * ea / n_obj
    g_b_obj = 2 * eb / n_obj

    # d sqrt(exp(raw)*anchor) / d raw = 0.5 * sqrt(pred_axis)
    a_unclipped = (raw_a[obj] > -6) & (raw_a[obj] < 2)
    b_unclipped = (raw_b[obj] > -6) & (raw_b[obj] < 2)
    g_a_obj += np.where(a_unclipped, esa * sqrt_pa / n_obj, 0.0)
    g_b_obj += np.where(b_unclipped, esb * sqrt_pb / n_obj, 0.0)
    g_a[obj] = g_a_obj
    g_b[obj] = g_b_obj
    grad[..., 3] = g_a * l_loc
    grad[..., 4] = g_b * l_loc

    loss_loc = loss_cx + loss_cy + loss_log_ab + loss_sqrt_ab

    # ── Angle θ: π-periodic smooth L1 ────────────────────────────────────
    theta_pred = np.pi * sigmoid(raw[..., 5])
    diff = theta_pred[obj] - t_th[obj]
    angle_err = np.minimum(np.abs(diff), np.pi - np.abs(diff))
    loss_theta = angle_err.mean()
    use_direct = np.abs(diff) <= (np.pi - np.abs(diff))
    sign = np.where(use_direct, np.sign(diff), -np.sign(diff))
    g_th = np.zeros_like(raw[..., 5])
    g_th[obj] = sign / n_obj * np.pi * sigmoid_grad(raw[..., 5][obj])
    grad[..., 5] = g_th * l_theta

    # ── Classification: softmax cross-entropy ─────────────────────────────
    cls_logits = raw[..., 6:]
    cls_pred   = cls_logits[obj]
    cls_target = t_cls[obj]
    probs      = softmax(cls_pred)
    loss_cls   = -np.log(probs[np.arange(n_obj), cls_target] + 1e-7).mean()
    g_cls      = np.zeros_like(cls_logits)
    g_cls_pred = probs.copy()
    g_cls_pred[np.arange(n_obj), cls_target] -= 1.0
    g_cls[obj] = g_cls_pred / n_obj
    grad[..., 6:] = g_cls * l_cls

    total = (l_loc   * loss_loc
           + l_obj   * loss_obj
           + l_noobj * loss_noobj
           + l_cls   * loss_cls
           + l_theta * loss_theta)

    return float(total), grad, dict(
        total=float(total), loc=float(loss_loc), obj=float(loss_obj),
        noobj=float(loss_noobj), cls=float(loss_cls), theta=float(loss_theta)
    )


## Section 6 — Training Loop

**Gradient clipping** prevents exploding gradients in early epochs when the
loss landscape is steep. We clip the global L2 norm of `grad_raw` to `GRAD_CLIP`.


In [ ]:
def make_batches(dataset, batch_size, shuffle=True):
    """Yield (img_batch [N,3,H,W], targets_list) mini-batches."""
    idx = np.arange(len(dataset))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch   = [dataset[i] for i in idx[start:start + batch_size]]
        imgs    = np.stack([b[0] for b in batch])
        targets = [b[1] for b in batch]
        yield imgs, targets


def train(model, train_ds, val_ds, epochs, batch_size, lr, save_dir, log_file,
          anchors=None, objectness_target=None):
    """
    Full training loop.
    - Constant LR Adam optimiser.
    - Global gradient-norm clipping.
    - Best-val-loss checkpoint saved each epoch.
    - Loss history written to JSON for later plotting.
    """
    os.makedirs(save_dir, exist_ok=True)
    anchors = ANCHORS if anchors is None else anchors
    objectness_target = OBJECTNESS_TARGET if objectness_target is None else objectness_target
    opt     = Adam(model.params(), lr=lr)
    history = []
    best_val = float("inf")
    keys = ["total", "loc", "obj", "noobj", "cls", "theta"]

    header = (f"{'Ep':>4} | {'Train':>8} | {'Val':>8} | "
              f"{'loc':>7} {'obj':>7} {'noobj':>7} {'cls':>7} {'θ':>7} | {'s':>5}")
    print(header); print("─" * len(header))

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # ── Train ──────────────────────────────────────────────────────────
        tr = {k: 0.0 for k in keys}
        nb = 0
        for imgs, tgts in make_batches(train_ds, batch_size, shuffle=True):
            raw = model.forward(imgs, training=True)
            loss, grad, info = ellipse_loss(
                raw, tgts, anchors, S, B, C,
                L_LOC, L_OBJ, L_NOOBJ, L_CLS, L_THETA,
                objectness_target)

            # Gradient clipping (global L2 norm)
            norm = np.sqrt((grad**2).sum())
            if norm > GRAD_CLIP:
                grad *= GRAD_CLIP / (norm + 1e-8)

            opt.zero_grad()
            model.backward(grad)
            opt.step()

            for k in keys:
                tr[k] += info.get(k, 0.0)
            nb += 1

        for k in keys:
            tr[k] /= max(nb, 1)

        # ── Validate (no weight update) ────────────────────────────────────
        vl_total = 0.0; vn = 0
        for imgs, tgts in make_batches(val_ds, batch_size, shuffle=False):
            raw = model.forward(imgs, training=False)
            _, _, info = ellipse_loss(
                raw, tgts, anchors, S, B, C,
                L_LOC, L_OBJ, L_NOOBJ, L_CLS, L_THETA,
                objectness_target)
            vl_total += info["total"]; vn += 1
        vl = vl_total / max(vn, 1)

        elapsed = time.time() - t0
        print(f"{epoch:>4} | {tr['total']:8.4f} | {vl:8.4f} | "
              f"{tr['loc']:7.4f} {tr['obj']:7.4f} {tr['noobj']:7.4f} "
              f"{tr['cls']:7.4f} {tr['theta']:7.4f} | {elapsed:5.1f}s")

        record = {"epoch": epoch, "train": tr, "val_total": float(vl)}
        history.append(record)

        if vl < best_val:
            best_val = vl
            model.save(f"{save_dir}/best.npz")

    with open(log_file, "w") as f:
        json.dump(history, f, indent=2)
    print(f"\nBest val loss: {best_val:.4f}   Log: {log_file}")
    return history


# ── Run training ───────────────────────────────────────────────────────────
model = EllipseNet(S=S, B=B, C=C)   # fresh model

history = train(
    model, train_ds, val_ds,
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE,
    lr         = LR,
    save_dir   = CHECKPOINT_DIR,
    log_file   = LOG_FILE,
)


## Section 7 — Training Curves


In [ ]:
def plot_history(history, save="figures/loss_curves.png"):
    if not history:
        print("No history to plot."); return
    epochs     = [r["epoch"] for r in history]
    train_tot  = [r["train"]["total"] for r in history]
    val_tot    = [r["val_total"]       for r in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(epochs, train_tot, marker="o", label="train total")
    ax1.plot(epochs, val_tot,   marker="o", label="val total")
    ax1.set(title="Total loss", xlabel="epoch", ylabel="loss")
    ax1.legend(); ax1.grid(alpha=0.3)

    for key, label in [("loc","loc"),("obj","obj"),
                        ("noobj","noobj"),("cls","cls"),("theta","θ")]:
        ax2.plot(epochs, [r["train"][key] for r in history],
                 marker="o", label=label)
    ax2.set(title="Train loss components", xlabel="epoch", ylabel="loss")
    ax2.legend(); ax2.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

plot_history(history)


## Section 8 — Evaluation: Precision, Recall, mAP

### Decoding predictions
For each grid cell `(i, j)` and anchor `b`, the detection is decoded as:
```
cx = (σ(cx_raw) + i) / S        # absolute horizontal centre
cy = (σ(cy_raw) + j) / S        # absolute vertical centre
a  = exp(log_a_raw) * anchor_a  # semi-major axis
b  = exp(log_b_raw) * anchor_b  # semi-minor axis
θ  = π · σ(θ_raw)               # orientation in [0, π)
score = σ(conf) · max(softmax(cls_logits))
```

### mAP computation
1. Collect all detections across the dataset with their confidence scores.
2. For each class, sort detections by confidence (descending).
3. Match each detection to a GT ellipse via **Ellipse-IoU ≥ 0.5**.  
   Each GT ellipse can only be matched once (greedy, highest-confidence first).
4. Compute the precision-recall curve and AP via **11-point interpolation**.
5. **mAP** = mean AP across classes that appear in the dataset.

> **Evaluation note:** we use ellipse-IoU (not box proxy) as the overlap metric,
> which is the best practice for an ellipse-based detector.


In [ ]:
def decode_single(raw, anchors, S, conf_thresh=0.5):
    """
    Decode one image's raw output (S, S, B, 6+C) into a flat list of detections.
    Each detection: (cx, cy, a, b, theta, score, class_id).
    Only returns detections whose joint score exceeds conf_thresh.
    """
    grid = np.arange(S)
    gx, gy = np.meshgrid(grid, grid)          # (S,S)

    conf   = sigmoid(raw[..., 0])             # (S,S,B)
    cx_off = sigmoid(raw[..., 1])
    cy_off = sigmoid(raw[..., 2])
    cx = (cx_off + gx[:, :, None]) / S
    cy = (cy_off + gy[:, :, None]) / S

    a = np.exp(np.clip(raw[..., 3], -6, 2)) * anchors[:, 0]
    b = np.exp(np.clip(raw[..., 4], -6, 2)) * anchors[:, 1]
    a = a.clip(1e-4, 0.5)
    b = b.clip(1e-4, 0.5)

    theta  = np.pi * sigmoid(raw[..., 5])
    cls_p  = softmax(raw[..., 6:])
    cls_id = cls_p.argmax(axis=-1)
    score  = conf * cls_p.max(axis=-1)

    mask = score > conf_thresh
    dets = []
    for i, j, bi in zip(*np.where(mask)):
        dets.append((float(cx[i,j,bi]), float(cy[i,j,bi]),
                     float(a[i,j,bi]),  float(b[i,j,bi]),
                     float(theta[i,j,bi]), float(score[i,j,bi]),
                     int(cls_id[i,j,bi])))
    return dets


def ellipse_nms(detections, iou_thresh=0.45, n_mc=150):
    """
    Non-maximum suppression for ellipse detections.
    Sorts by score (descending) and suppresses overlapping boxes.
    """
    if not detections:
        return []
    dets = sorted(detections, key=lambda d: -d[5])
    keep = []
    while dets:
        best = dets.pop(0)
        keep.append(best)
        e1 = best[:5]
        dets = [d for d in dets
                if ellipse_iou_mc(e1, d[:5], n=n_mc) < iou_thresh]
    return keep


def evaluate(model, dataset, anchors, S, B, C,
             conf_thresh=0.5, iou_thresh=0.5, n_mc=200):
    """
    Compute per-class AP and overall mAP using ellipse-IoU matching.

    Algorithm
    ---------
    1. For each image: decode predictions, apply NMS, match to GT.
    2. Label each detection TP (IoU ≥ iou_thresh, correct class, unmatched GT)
       or FP (otherwise).
    3. Per class: sort by score, compute cumulative precision/recall, integrate
       AP with 11-point interpolation.
    4. mAP = mean(AP over classes that have at least one GT instance).
    """
    preds_by_cls = {c: [] for c in range(C)}    # class → [(score, is_tp)]
    n_gt_by_cls  = {c: 0  for c in range(C)}

    for idx in range(len(dataset)):
        img, gts = dataset[idx]

        # Count GT instances per class
        for gt in gts:
            n_gt_by_cls[gt["class_id"]] += 1

        # Get predictions for this image
        raw  = model.forward(img[None], training=False)[0]   # (S,S,B,6+C)
        dets = ellipse_nms(decode_single(raw, anchors, S, conf_thresh))

        # Greedy matching: highest-confidence detections get first pick of GT
        matched_gt = [False] * len(gts)

        for det in sorted(dets, key=lambda d: -d[5]):
            cx, cy, a, b, θ, score, cls_id = det
            best_iou, best_j = 0.0, -1

            for j, gt in enumerate(gts):
                if matched_gt[j] or gt["class_id"] != cls_id:
                    continue
                iou = ellipse_iou_mc(
                    (cx, cy, a, b, θ),
                    (gt["cx"], gt["cy"], gt["a"], gt["b"], gt["theta"]),
                    n=n_mc)
                if iou > best_iou:
                    best_iou, best_j = iou, j

            is_tp = (best_iou >= iou_thresh and best_j >= 0)
            if is_tp:
                matched_gt[best_j] = True
            preds_by_cls[cls_id].append((score, float(is_tp)))

    # ── Compute AP per class ───────────────────────────────────────────────
    aps = {}
    for c in range(C):
        if n_gt_by_cls[c] == 0:
            continue
        entries = sorted(preds_by_cls[c], key=lambda x: -x[0])
        if not entries:
            aps[VOC_CLASSES[c]] = 0.0
            continue
        tp_cum = np.cumsum([e[1] for e in entries])
        fp_cum = np.cumsum([1 - e[1] for e in entries])
        prec   = tp_cum / (tp_cum + fp_cum + 1e-8)
        rec    = tp_cum / (n_gt_by_cls[c] + 1e-8)

        # 11-point interpolated AP (Pascal VOC protocol)
        ap = 0.0
        for t in np.linspace(0.0, 1.0, 11):
            p_at_t = prec[rec >= t].max() if (rec >= t).any() else 0.0
            ap += p_at_t / 11.0
        aps[VOC_CLASSES[c]] = ap

    mAP = float(np.mean(list(aps.values()))) if aps else 0.0

    # ── Micro-averaged precision & recall across all classes ──────────────
    all_preds = [(s, t) for c in range(C) for s, t in preds_by_cls[c]]
    gt_total  = sum(n_gt_by_cls.values())
    if all_preds:
        tp_total = sum(t for _, t in all_preds)
        precision = tp_total / max(len(all_preds), 1)
        recall    = tp_total / max(gt_total, 1)
    else:
        precision = recall = 0.0

    return {
        "mAP":           round(mAP,       4),
        "precision":     round(precision, 4),
        "recall":        round(recall,    4),
        "per_class_AP":  {k: round(v, 4) for k, v in aps.items()},
        "n_gt_by_cls":   {VOC_CLASSES[c]: n for c, n in n_gt_by_cls.items() if n > 0},
    }


def threshold_sweep(model, dataset, anchors, S, B, C,
                    thresholds=CONF_THRESHOLDS, iou_thresh=EVAL_IOU_THRESH):
    """Print mAP/precision/recall across confidence thresholds."""
    rows = []
    print("\nConfidence threshold sweep:")
    print(f"{'conf':>6} {'mAP':>8} {'prec':>8} {'recall':>8}")
    for conf in thresholds:
        m = evaluate(model, dataset, anchors, S, B, C,
                     conf_thresh=conf, iou_thresh=iou_thresh)
        rows.append((conf, m))
        print(f"{conf:6.2f} {m['mAP']:8.4f} {m['precision']:8.4f} {m['recall']:8.4f}")
    return rows


# ── Load best checkpoint and evaluate ─────────────────────────────────────
ckpt = f"{CHECKPOINT_DIR}/best.npz"
if os.path.exists(ckpt):
    model.load(ckpt)

print("Evaluating on training set (overfit check)...")
metrics_train = evaluate(model, train_ds, ANCHORS, S, B, C,
                          conf_thresh=DEFAULT_CONF_THRESH, iou_thresh=EVAL_IOU_THRESH)
print(f"  mAP       : {metrics_train['mAP']:.4f}")
print(f"  Precision : {metrics_train['precision']:.4f}")
print(f"  Recall    : {metrics_train['recall']:.4f}")

print("\nEvaluating on validation set...")
metrics_val = evaluate(model, val_ds, ANCHORS, S, B, C,
                        conf_thresh=DEFAULT_CONF_THRESH, iou_thresh=EVAL_IOU_THRESH)
print(f"  mAP       : {metrics_val['mAP']:.4f}")
print(f"  Precision : {metrics_val['precision']:.4f}")
print(f"  Recall    : {metrics_val['recall']:.4f}")

# Per-class breakdown
if metrics_val["per_class_AP"]:
    print("\nPer-class AP (val):")
    for cls_name, ap in sorted(metrics_val["per_class_AP"].items(),
                                key=lambda x: -x[1]):
        n = metrics_val["n_gt_by_cls"].get(cls_name, 0)
        print(f"  {cls_name:<15} AP={ap:.3f}  (GT instances: {n})")

threshold_rows_val = threshold_sweep(model, val_ds, ANCHORS, S, B, C)


## Section 9 — Prediction Visualisation

- **Lime dashed** ellipses = ground truth  
- **Red solid** ellipses = model predictions  

Failure modes to inspect: missed detections (lime without red), false positives
(red without lime), tight-fit quality, orientation correctness.


In [ ]:
def visualize_predictions(model, dataset, anchors, S, n=6,
                          conf_thresh=DEFAULT_CONF_THRESH, nms_thresh=0.45,
                          save="figures/predictions.png"):
    """
    Draw ground-truth (dashed lime) and predicted (solid red) ellipses
    side-by-side with their confidence scores and class labels.
    """
    cols, rows = 3, (n + 2) // 3
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    for ax in axes.flat:
        ax.axis("off")

    for ax, idx in zip(axes.flat, range(min(n, len(dataset)))):
        img, gts = dataset[idx]
        ax.imshow(img.transpose(1, 2, 0).clip(0, 1))

        # Ground truth (lime dashed)
        for gt in gts:
            ax.add_patch(MplEllipse(
                (gt["cx"] * IMG_SIZE, gt["cy"] * IMG_SIZE),
                2 * gt["a"] * IMG_SIZE, 2 * gt["b"] * IMG_SIZE,
                angle=np.rad2deg(gt["theta"]),
                fill=False, edgecolor="lime", lw=1.2, linestyle="--"))

        # Predictions (red solid)
        raw  = model.forward(img[None], training=False)[0]
        dets = ellipse_nms(decode_single(raw, anchors, S, conf_thresh),
                           iou_thresh=nms_thresh)
        for det in dets:
            cx, cy, a, b, θ, score, cls_id = det
            ax.add_patch(MplEllipse(
                (cx * IMG_SIZE, cy * IMG_SIZE),
                2 * a * IMG_SIZE, 2 * b * IMG_SIZE,
                angle=np.rad2deg(θ),
                fill=False, edgecolor="red", lw=1.5))
            ax.text(cx * IMG_SIZE, cy * IMG_SIZE,
                    f"{VOC_CLASSES[cls_id]} {score:.2f}",
                    color="white", fontsize=6, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.1", fc="red", alpha=0.5))

        ax.set_title(f"sample {idx}", fontsize=9)

    fig.suptitle("Lime dashed = GT   |   Red solid = Prediction", fontsize=10)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

visualize_predictions(model, train_ds, ANCHORS, S,
                       n=6, conf_thresh=DEFAULT_CONF_THRESH, save="figures/train_preds.png")


## Section 10 — Optional Hyperparameter Experiments

These cells are disabled by default. Use them after the main pipeline works to compare learning rates, confidence thresholds, and a tiny overfit run without overwriting the main Final checkpoint.


In [ ]:
# ── Optional experiment controls ───────────────────────────────────────────
RUN_HPARAM_EXPERIMENTS = False
RUN_OVERFIT_EXPERIMENT = False

EXP_MAX_TRAIN = 100
EXP_MAX_VAL = 100
EXP_EPOCHS = 5
EXP_BATCH_SIZE = 4
EXP_LRS = [1e-3, 5e-4, 1e-4]
EXP_THRESHOLDS = [0.3, 0.5, 0.7, 0.8]
EXP_ROOT = "checkpoints_final_experiments"
EXP_LOG = "outputs/hparam_summary_final.json"


def experiment_name_for_lr(lr):
    """Make a filesystem-safe name such as lr_0p0005."""
    return f"lr_{lr:g}".replace(".", "p").replace("-", "m")


def best_val_from_history(history):
    """Return the lowest validation loss recorded by train(...)."""
    if not history:
        return float("nan")
    return float(min(row["val_total"] for row in history))


def print_experiment_table(rows):
    """Compact summary for LR/threshold/overfit experiments."""
    if not rows:
        print("No experiment rows to show.")
        return
    header = (f"{'experiment':<18} {'lr':>9} {'best_val':>10} "
              f"{'conf':>6} {'mAP':>8} {'prec':>8} {'recall':>8}")
    print(header)
    print("─" * len(header))
    for row in rows:
        print(f"{row['experiment']:<18} {row.get('lr', float('nan')):>9g} "
              f"{row.get('best_val_loss', float('nan')):>10.4f} "
              f"{row['conf_thresh']:>6.2f} {row['mAP']:>8.4f} "
              f"{row['precision']:>8.4f} {row['recall']:>8.4f}")


def evaluate_thresholds_for_experiment(model, dataset, anchors, thresholds):
    """Evaluate one trained model across confidence thresholds."""
    rows = []
    for conf in thresholds:
        metrics = evaluate(model, dataset, anchors, S, B, C,
                           conf_thresh=conf, iou_thresh=EVAL_IOU_THRESH)
        rows.append({
            "conf_thresh": float(conf),
            "mAP": float(metrics["mAP"]),
            "precision": float(metrics["precision"]),
            "recall": float(metrics["recall"]),
        })
    return rows


def run_lr_experiments(max_train=EXP_MAX_TRAIN, max_val=EXP_MAX_VAL,
                       epochs=EXP_EPOCHS, batch_size=EXP_BATCH_SIZE,
                       lrs=EXP_LRS, thresholds=EXP_THRESHOLDS,
                       root=EXP_ROOT, log_file=EXP_LOG,
                       objectness_target=OBJECTNESS_TARGET):
    """
    Train a fresh staged model for each LR, then evaluate each best checkpoint
    across confidence thresholds. This is intentionally small and optional.
    """
    os.makedirs(root, exist_ok=True)
    exp_train = VOCEllipseDataset(
        voc_root, "train", max_train,
        random_rotation=TRAIN_RANDOM_ROTATION,
        rotation_range=ROTATION_RANGE_DEG,
    )
    exp_val = VOCEllipseDataset(voc_root, "val", max_val, random_rotation=False)
    exp_anchors = estimate_anchors_from_samples(exp_train.samples, B)

    print(f"Experiment data: train={len(exp_train)} val={len(exp_val)}")
    print(f"Experiment anchors: {np.round(exp_anchors, 4).tolist()}")

    summary = []
    for lr in lrs:
        name = experiment_name_for_lr(lr)
        save_dir = os.path.join(root, name)
        print(f"\n=== LR experiment: {lr:g} ===")

        exp_model = EllipseNet(S=S, B=B, C=C)
        history_exp = train(
            exp_model, exp_train, exp_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            save_dir=save_dir,
            log_file=os.path.join(save_dir, "train_log.json"),
            anchors=exp_anchors,
            objectness_target=objectness_target,
        )

        ckpt = os.path.join(save_dir, "best.npz")
        if os.path.exists(ckpt):
            exp_model.load(ckpt)

        threshold_rows = evaluate_thresholds_for_experiment(
            exp_model, exp_val, exp_anchors, thresholds)
        best_val_loss = best_val_from_history(history_exp)
        best_threshold_row = max(threshold_rows, key=lambda row: row["mAP"])

        for row in threshold_rows:
            summary.append({
                "experiment": name,
                "lr": float(lr),
                "best_val_loss": best_val_loss,
                "objectness_target": objectness_target,
                **row,
            })

        print("Best threshold by mAP:", best_threshold_row)

    with open(log_file, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nExperiment summary saved → {log_file}")
    print_experiment_table(summary)
    return summary


def run_overfit_experiment(epochs=EXP_EPOCHS, batch_size=EXP_BATCH_SIZE,
                           lr=5e-4, thresholds=EXP_THRESHOLDS,
                           root=EXP_ROOT,
                           objectness_target=OBJECTNESS_TARGET):
    """
    Tiny memorization check: 20 train samples are used for both train and val,
    with rotation disabled so the target is deterministic.
    """
    save_dir = os.path.join(root, "overfit_debug")
    os.makedirs(save_dir, exist_ok=True)

    overfit_train = VOCEllipseDataset(voc_root, "train", 20, random_rotation=False)
    overfit_val = VOCEllipseDataset(voc_root, "train", 20, random_rotation=False)
    overfit_anchors = estimate_anchors_from_samples(overfit_train.samples, B)

    print(f"Overfit data: train={len(overfit_train)} val={len(overfit_val)}")
    print(f"Overfit anchors: {np.round(overfit_anchors, 4).tolist()}")

    overfit_model = EllipseNet(S=S, B=B, C=C)
    history_overfit = train(
        overfit_model, overfit_train, overfit_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        save_dir=save_dir,
        log_file=os.path.join(save_dir, "train_log.json"),
        anchors=overfit_anchors,
        objectness_target=objectness_target,
    )

    ckpt = os.path.join(save_dir, "best.npz")
    if os.path.exists(ckpt):
        overfit_model.load(ckpt)

    threshold_rows = evaluate_thresholds_for_experiment(
        overfit_model, overfit_train, overfit_anchors, thresholds)
    summary = [{
        "experiment": "overfit_debug",
        "lr": float(lr),
        "best_val_loss": best_val_from_history(history_overfit),
        "objectness_target": objectness_target,
        **row,
    } for row in threshold_rows]

    print_experiment_table(summary)
    return summary


hparam_summary = []
if RUN_HPARAM_EXPERIMENTS:
    hparam_summary.extend(run_lr_experiments())

if RUN_OVERFIT_EXPERIMENT:
    hparam_summary.extend(run_overfit_experiment())

if hparam_summary:
    print("\nCombined experiment results:")
    print_experiment_table(hparam_summary)
else:
    print("Optional experiments are disabled. Set RUN_HPARAM_EXPERIMENTS or RUN_OVERFIT_EXPERIMENT to True to run them.")


### Section 10.1 — Plot Hyperparameter Test Runs

Plot the saved learning-rate experiment logs and threshold-sweep summary. Run this after the optional hyperparameter experiments have produced `outputs/hparam_summary_final.json`.


In [ ]:
def load_json_if_exists(path):
    """Load JSON if it exists; return None otherwise."""
    if not os.path.exists(path):
        print(f"Missing file: {path}")
        return None
    with open(path) as f:
        return json.load(f)


def plot_experiment_train_logs(root=EXP_ROOT, lrs=EXP_LRS,
                               save="figures/final_lr_experiment_losses.png"):
    """Plot train/val loss curves for each LR experiment."""
    fig, ax = plt.subplots(figsize=(8, 5))
    plotted = False

    for lr in lrs:
        name = experiment_name_for_lr(lr)
        log_path = os.path.join(root, name, "train_log.json")
        hist = load_json_if_exists(log_path)
        if not hist:
            continue
        epochs = [row["epoch"] for row in hist]
        train_loss = [row["train"]["total"] for row in hist]
        val_loss = [row["val_total"] for row in hist]
        ax.plot(epochs, train_loss, marker="o", linestyle="-",  label=f"lr={lr:g} train")
        ax.plot(epochs, val_loss,   marker="o", linestyle="--", label=f"lr={lr:g} val")
        plotted = True

    if not plotted:
        print("No LR experiment logs found to plot.")
        plt.close(fig)
        return

    ax.set(title="Learning-rate experiment losses", xlabel="epoch", ylabel="loss")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")


def plot_threshold_summary(summary_path=EXP_LOG,
                           save="figures/final_threshold_summary.png"):
    """Plot mAP/precision/recall from the hyperparameter threshold sweep."""
    rows = load_json_if_exists(summary_path)
    if not rows:
        return

    lr_rows = [row for row in rows if row["experiment"].startswith("lr_")]
    if not lr_rows:
        print("No LR threshold rows found in summary.")
        return

    experiments = sorted(set(row["experiment"] for row in lr_rows))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True)
    metrics = [("mAP", "mAP"), ("precision", "precision"), ("recall", "recall")]

    for exp in experiments:
        sub = sorted([row for row in lr_rows if row["experiment"] == exp],
                     key=lambda row: row["conf_thresh"])
        xs = [row["conf_thresh"] for row in sub]
        for ax, (key, title) in zip(axes, metrics):
            ax.plot(xs, [row[key] for row in sub], marker="o", label=exp)
            ax.set(title=title, xlabel="confidence threshold")
            ax.grid(alpha=0.3)

    axes[0].set_ylabel("score")
    axes[-1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")


plot_experiment_train_logs()
plot_threshold_summary()


### Section 10.2 — Overfitting Experiment

This is a deterministic 20-image memorization run. It uses 20 training images for both train and validation, trains for 20 epochs with `LR=5e-4`, then plots the overfit loss curve.


In [ ]:
# ── Overfitting run: 20 images, 20 epochs, LR=5e-4 ─────────────────────────
OVERFIT_IMAGES = 20
OVERFIT_EPOCHS = 20
OVERFIT_LR = 5e-4
OVERFIT_CONF_THRESH = 0.7
OVERFIT_SAVE_DIR = os.path.join(EXP_ROOT, "overfitting")
OVERFIT_LOG = os.path.join(OVERFIT_SAVE_DIR, "train_log.json")

os.makedirs(OVERFIT_SAVE_DIR, exist_ok=True)

overfit_train_ds = VOCEllipseDataset(voc_root, "train", OVERFIT_IMAGES, random_rotation=False)
overfit_val_ds = VOCEllipseDataset(voc_root, "train", OVERFIT_IMAGES, random_rotation=False)
overfit_anchors = estimate_anchors_from_samples(overfit_train_ds.samples, B)

overfit_model = EllipseNet(S=S, B=B, C=C)
overfit_history = train(
    overfit_model, overfit_train_ds, overfit_val_ds,
    epochs=OVERFIT_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=OVERFIT_LR,
    save_dir=OVERFIT_SAVE_DIR,
    log_file=OVERFIT_LOG,
    anchors=overfit_anchors,
    objectness_target=OBJECTNESS_TARGET,
)

plot_history(overfit_history, save="figures/final_overfitting_loss.png")

ckpt = os.path.join(OVERFIT_SAVE_DIR, "best.npz")
if os.path.exists(ckpt):
    overfit_model.load(ckpt)

overfit_metrics = evaluate(
    overfit_model, overfit_train_ds, overfit_anchors, S, B, C,
    conf_thresh=OVERFIT_CONF_THRESH,
    iou_thresh=EVAL_IOU_THRESH,
)
print("Overfitting metrics:")
print(f"  confidence : {OVERFIT_CONF_THRESH}")
print(f"  mAP        : {overfit_metrics['mAP']:.4f}")
print(f"  precision  : {overfit_metrics['precision']:.4f}")
print(f"  recall     : {overfit_metrics['recall']:.4f}")


### Section 10.3 — Overfitting Predictions

Visualize the best checkpoint from the overfitting run. The default confidence threshold is `0.7`; lower it to `0.3` if the model learned but confidence calibration is still conservative.


In [ ]:
# ── Visualize predictions from the overfitting checkpoint ──────────────────
ckpt = os.path.join(OVERFIT_SAVE_DIR, "best.npz")
if os.path.exists(ckpt):
    overfit_model.load(ckpt)
else:
    print(f"Missing checkpoint: {ckpt}. Run the overfitting cell first.")

visualize_predictions(
    overfit_model,
    overfit_train_ds,
    overfit_anchors,
    S,
    n=6,
    conf_thresh=OVERFIT_CONF_THRESH,
    save="figures/final_overfitting_predictions.png",
)


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print("=" * 55)
print("  EllipseNet — Final Results Summary")
print("=" * 55)
print(f"  Dataset    : Pascal VOC 2012")
print(f"  Input size : {IMG_SIZE}×{IMG_SIZE}")
print(f"  Grid       : {S}×{S}  |  {B} anchors  |  {C} classes")
print(f"  Parameters : {sum(p.size for p, _ in model.params()):,}")
print(f"  Epochs     : {EPOCHS}  |  LR={LR}  |  Batch={BATCH_SIZE}")
print(f"  Anchors    : {np.round(ANCHORS, 4).tolist()}")
print(f"  Obj target : {OBJECTNESS_TARGET}")
print()
print("  Training set (overfit check):")
print(f"    mAP       : {metrics_train['mAP']:.4f}")
print(f"    Precision : {metrics_train['precision']:.4f}")
print(f"    Recall    : {metrics_train['recall']:.4f}")
print()
print("  Validation set:")
print(f"    mAP       : {metrics_val['mAP']:.4f}")
print(f"    Precision : {metrics_val['precision']:.4f}")
print(f"    Recall    : {metrics_val['recall']:.4f}")
print("=" * 55)
